In [1]:
import pandas as pd
import numpy as np
import nibabel as nib
from glob import glob
import os
from tqdm.notebook import tqdm
import torch

In [2]:
def iou_score(pred, mask, smoothing=1e-8):
    pred = pred.reshape(-1)
    mask = mask.reshape(-1)
    
    interaction = torch.sum(pred * mask)
    denominator = torch.count_nonzero(pred + mask)

    iou = (interaction+smoothing)/(denominator+smoothing)
    return iou

In [17]:
for gt_path in glob("/data/datasets/nii/MSD/Task03_Liver/labelsTs/*"):
    gt = nib.load(gt_path)
    gt = np.asarray(gt.dataobj, dtype=np.float32)
    
    gt = torch.tensor(gt).permute(2,0,1)
    
    for obj_id in torch.unique(gt)[1:]:
        obj_gt = torch.where(gt == obj_id, 1, 0)
        
        pos = torch.argwhere(obj_gt.sum(dim=(1,2)) > 0)
        
        out_bank, in_bank = 0, 0
        
        for frame_idx in pos:
            iou_list, idx_list = [], []
            for prev_idx in pos:
                if prev_idx >= frame_idx:
                    break
                
                iou = iou_score(obj_gt[frame_idx], obj_gt[prev_idx])
                iou_list.append(iou)
                idx_list.append(prev_idx)
            
            if len(iou_list) > 6:
                iou_list = torch.tensor(iou_list)
                min_inbank = iou_list[-6:].min()
                max_outbank = iou_list[:-6].max()
                
                if min_inbank < max_outbank:
                    out_bank += 1
                else:
                    in_bank += 1
                
        print(gt_path, obj_id.item(), out_bank, in_bank)
                

/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_92.nii.gz 1.0 0 172
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_92.nii.gz 2.0 32 6
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_60.nii.gz 1.0 10 62
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_60.nii.gz 2.0 4 7
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_34.nii.gz 1.0 0 91
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_70.nii.gz 1.0 1 102
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_70.nii.gz 2.0 15 29
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_125.nii.gz 1.0 0 105
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_125.nii.gz 2.0 0 2
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_88.nii.gz 1.0 0 168
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_88.nii.gz 2.0 21 74
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_76.nii.gz 1.0 3 65
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_76.nii.gz 2.0 33 19
/data/datasets/nii/MSD/Task03_Liver/labelsTs/liver_30.nii.gz 1.0 0 116
/data/datas